In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_ta as ta
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

In [ ]:
def import_data(stock, startDate):
    data = yf.download(stock, start=startDate, interval='1d')
    data.columns = data.columns.get_level_values(0)

    print(data)

    data['EMA_200'] = ta.ema(data['Close'], length=200)
    data['SMA_50'] = ta.sma(data['Close'], length=50)
    data['SMA_200'] = ta.sma(data['Close'], length=200)
    data['RSI'] = ta.rsi(data['Close'], length=14)
    data['RSI_5'] = ta.rsi(data['Close'], length=5)
    data['ATR'] = ta.atr(data['High'], data['Low'], data['Close'], length=14)
    # data['Engulfing'] = data.ta.cdl_pattern(name="engulfing")
    bbands = ta.bbands(data['Close'], length=20, std=2)
    macd = ta.macd(data['Close'])
    data = data.join(bbands)
    data = data.join(macd)

    return data

In [ ]:
# %% 1. Download and Prepare the Data
# We use BTC-USD from yfinance. For a baseline, we focus on Bitcoin.
symbol = "BTC-USD"
start_date = "2017-01-01"
end_date = "2025-05-14"
df = import_data(symbol, start_date)

# Create a target variable based on next day return
# Calculate the next-period return and assign 1 if positive, 0 otherwise.
df['Return'] = df['Close'].pct_change().shift(-1)
df['Target'] = (df['Return'] > 0).astype(int)

# Drop any rows with missing values
df.dropna(inplace=True)

# %% 2. Feature Selection and Scaling
# For our features we use price data and technical indicators.
feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume', 
    'RSI', 'SMA_50', 'SMA_200', 'EMA_200', 'MACD_12_26_9', 
    'BBL_20_2.0', 'BBM_20_2.0', 'BBU_20_2.0'
]

# It’s a good idea to scale the features.
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[feature_cols] = scaler.fit_transform(df[feature_cols])

# %% 3. Create Sequences for Time Series Modeling
# We build sequences of a fixed window (e.g., 60 days)
def create_sequences(df, seq_length, feature_cols):
    X, y, ret = [], [], []
    for i in range(len(df) - seq_length):
        seq_data = df[feature_cols].iloc[i:i+seq_length].values
        # The target corresponds to the next day after the window.
        target = df['Target'].iloc[i+seq_length]
        ret_val = df['Return'].iloc[i+seq_length]
        X.append(seq_data)
        y.append(target)
        ret.append(ret_val)
    return np.array(X), np.array(y), np.array(ret)

seq_length = 60
X, y, returns_arr = create_sequences(df_scaled, seq_length, feature_cols)

# %% 4. Split Data into Training and Test Sets
# We will use the first 80% of sequences for training and the remaining for testing.
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]
returns_test = returns_arr[train_size:]

# %% 5. Build the LSTM and GRU Models
def create_lstm_model(input_shape):
    model = Sequential()
    model.add(LSTM(50, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(LSTM(50))
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='sigmoid'))  # Sigmoid outputs a probability for binary classification
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

def create_gru_model(input_shape):
    model = Sequential()
    model.add(GRU(50, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(GRU(50))
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

input_shape = (X_train.shape[1], X_train.shape[2])
lstm_model = create_lstm_model(input_shape)
gru_model = create_gru_model(input_shape)

# %% 6. Train the Models Using Walk-Forward (Time-Series) Validation
# Here we use a validation split (last 10% of the training set) and early stopping.
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("Training LSTM model...")
lstm_history = lstm_model.fit(X_train, y_train, epochs=50, batch_size=32, 
                              validation_split=0.1, callbacks=[early_stop], verbose=1)

print("Training GRU model...")
gru_history = gru_model.fit(X_train, y_train, epochs=50, batch_size=32, 
                            validation_split=0.1, callbacks=[early_stop], verbose=1)

# %% 7. Evaluate Models on the Test Set
lstm_preds = lstm_model.predict(X_test)
gru_preds = gru_model.predict(X_test)

# %% 8. Testing Strategy Effectiveness by Optimizing the Buy/Sell Probability Threshold
# For our trading simulation, we assume:
#   - if the predicted probability is above a threshold, we "go long" (bet on an up move)
#   - if below the threshold, we take a contrarian (or short) position.
# We then simulate strategy performance by applying the actual next-day return:
#   - If prediction = 1, take the actual return.
#   - If prediction = 0, assume shorting so we get the inverse of the actual return.

def strategy_performance(pred_probs, actual_returns, true_labels, threshold):
    # Convert probabilities into binary decisions
    preds_class = (pred_probs > threshold).astype(int).flatten()
    # Simulated strategy: if signal is 1, we capture the actual return;
    # if signal is 0, we “short” (i.e. take negative of the return).
    strat_returns = np.where(preds_class == 1, actual_returns, -actual_returns)
    # Compute cumulative return multiplicatively over the periods.
    cumulative_return = np.prod(1 + strat_returns) - 1
    accuracy = np.mean(preds_class == true_labels)
    return cumulative_return, accuracy

thresholds = np.linspace(0.1, 0.9, 9)
print("\nEvaluating thresholds on the test set:")

print("\nLSTM Model Performance:")
for thresh in thresholds:
    cum_return, acc = strategy_performance(lstm_preds, returns_test, y_test, thresh)
    print(f"Threshold {thresh:.2f}: Cumulative return: {cum_return:.4f}, Accuracy: {acc:.4f}")

print("\nGRU Model Performance:")
for thresh in thresholds:
    cum_return, acc = strategy_performance(gru_preds, returns_test, y_test, thresh)
    print(f"Threshold {thresh:.2f}: Cumulative return: {cum_return:.4f}, Accuracy: {acc:.4f}")

# %% 9. (Optional) Plotting the Training History for Each Model
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(lstm_history.history['loss'], label='Train Loss')
plt.plot(lstm_history.history['val_loss'], label='Val Loss')
plt.title('LSTM Loss History')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(gru_history.history['loss'], label='Train Loss')
plt.plot(gru_history.history['val_loss'], label='Val Loss')
plt.title('GRU Loss History')
plt.legend()
plt.show()


**NEW**

In [1]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta

from backtesting import Backtest, Strategy

# Original import_data function remains unchanged
def import_data(stock, startDate):
    df = yf.download(stock, start=startDate, interval='1h')
    df.columns = df.columns.get_level_values(0)

    print(df)

    # 200-period EMA on Close
    df["EMA200"] = ta.ema(df["Close"], length=200)

    # Heikin-Ashi candles
    ha = ta.ha(df["Open"], df["High"], df["Low"], df["Close"])
    # ha returns a DataFrame with columns: HA_open, HA_high, HA_low, HA_close
    df = pd.concat([df, ha], axis=1)

    # Parabolic SAR (long and short lines)
    psar = ta.psar(df["High"], df["Low"], df["Close"])
    # PSARl = long-side SAR; PSARs = short-side SAR
    df["PSARl"] = psar["PSARl_0.02_0.2"]
    df["PSARs"] = psar["PSARs_0.02_0.2"]

    # 14-period RSI
    df["RSI14"] = ta.rsi(df["Close"], length=14)

    # 14-period ATR for volatility-based stops
    df["ATR14"] = ta.atr(df["High"], df["Low"], df["Close"], length=14)
    return df

c:\Users\cdpea\miniforge3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\cdpea\miniforge3\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
symbol     = "BTC-USD"
start_date = "2023-05-20"
interval   = "15m"

df = import_data(symbol, start_date)

print(df)
# Drop any rows with missing data after download
# df.dropna(inplace=True)

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed

1 Failed download:
['BTC-USD']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


Empty DataFrame
Columns: [Adj Close, Close, High, Low, Open, Volume]
Index: []


IndexError: single positional indexer is out-of-bounds

In [48]:
class CryptoPSARStrategy(Strategy):
    """
    Hybrid trend + momentum + volatility strategy:
     - Trend filter: Price vs EMA200
     - Entry: Heikin Ashi candle flip + PSAR flip + RSI filter
     - Stop: 2 × ATR14
     - Exit: PSAR reversal trailing stop
    """
    def init(self):
        # Cache series for speed
        self.ema200 = self.data["EMA200"]
        self.ha_open = self.data["HA_open"]
        self.ha_close = self.data["HA_close"]
        self.psar_l = self.data["PSARl"]
        self.psar_s = self.data["PSARs"]
        self.rsi14  = self.data["RSI14"]
        self.atr14  = self.data["ATR14"]

    def next(self):
        price = self.data.Close[-1]

        # If no open position, check for entry
        if not self.position:
            # LONG ENTRY conditions
            cond_trend_up    = price > self.ema200[-1]
            cond_ha_bull     = self.ha_close[-1] > self.ha_open[-1]
            # PSAR flip: previous bar PSAR > price && current PSAR < price
            cond_psar_flip_l = (self.psar_l[-2] > self.data.Close[-2]) and (self.psar_l[-1] < price)
            cond_rsi_ok      = self.rsi14[-1] < 50

            if cond_trend_up and cond_ha_bull and cond_psar_flip_l and cond_rsi_ok:
                # Set stop-loss 2 × ATR below entry
                stop_price = price - 2 * self.atr14[-1]
                self.buy(sl=stop_price)

            # SHORT ENTRY conditions (mirror logic)
            cond_trend_dn    = price < self.ema200[-1]
            cond_ha_bear     = self.ha_close[-1] < self.ha_open[-1]
            cond_psar_flip_s = (self.psar_s[-2] < self.data.Close[-2]) and (self.psar_s[-1] > price)
            cond_rsi_ok_s    = self.rsi14[-1] > 50

            if cond_trend_dn and cond_ha_bear and cond_psar_flip_s and cond_rsi_ok_s:
                stop_price = price + 2 * self.atr14[-1]
                self.sell(sl=stop_price)

        # If in a position, exit on PSAR reversal
        else:
            if self.position.is_long:
                # PSAR dot flips above price → exit long
                if self.psar_l[-1] > price:
                    self.position.close()
            else:
                # PSAR dot flips below price → exit short
                if self.psar_s[-1] < price:
                    self.position.close()

In [51]:
print(df)

bt = Backtest(
    df,
    CryptoPSARStrategy,
    cash=10_000,
    commission=0.0005,   # ~0.05% per trade side
    trade_on_close=True, # execute orders at close price of signal bar
    # exclusive=True       # only one position at a time
)

stats = bt.run()
print(stats)
bt.plot()  # opens an interactive equity curve & trade chart


C:\Users\cdpea\AppData\Local\Temp\ipykernel_24448\2193375303.py:3: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (GH-134).
  bt = Backtest(


                                   Close           High            Low  \
Datetime                                                                 
2023-05-20 00:00:00+00:00   26856.328125   26902.261719   26843.277344   
2023-05-20 01:00:00+00:00   26869.582031   26889.425781   26856.148438   
2023-05-20 02:00:00+00:00   26851.123047   26881.185547   26845.404297   
2023-05-20 03:00:00+00:00   26873.246094   26880.650391   26848.378906   
2023-05-20 04:00:00+00:00   26878.585938   26878.585938   26861.433594   
...                                  ...            ...            ...   
2025-05-17 17:00:00+00:00  103165.062500  103311.484375  102957.882812   
2025-05-17 18:00:00+00:00  103003.968750  103271.125000  103003.968750   
2025-05-17 19:00:00+00:00  103207.406250  103207.406250  102963.296875   
2025-05-17 20:00:00+00:00  103178.257812  103218.609375  103118.046875   
2025-05-18 00:00:00+00:00  103253.140625  103253.140625  103142.601562   

                                    O

c:\Users\cdpea\miniforge3\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '2h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


Start                     2023-05-20 00:00...
End                       2025-05-18 00:00...
Duration                    729 days 00:00:00
Exposure Time [%]                         0.0
Equity Final [$]                      10000.0
Equity Peak [$]                       10000.0
Return [%]                                0.0
Buy & Hold Return [%]               284.46485
Return (Ann.) [%]                         0.0
Volatility (Ann.) [%]                     0.0
CAGR [%]                                  0.0
Sharpe Ratio                              NaN
Sortino Ratio                             NaN
Calmar Ratio                              NaN
Alpha [%]                                 0.0
Beta                                      0.0
Max. Drawdown [%]                        -0.0
Avg. Drawdown [%]                         NaN
Max. Drawdown Duration                    NaN
Avg. Drawdown Duration                    NaN
# Trades                                    0
Win Rate [%]                      

GridPlot(id='p1157', ...)